In [4]:
# =========================================================
# FATE FRACTIONS per angle, for each alpha
# =========================================================
import numpy as np
import pandas as pd

CSV_FILE = "particles_all_runs.csv"
ALPHAS   = [1e2, 1e3, 1e4,1e5]

df = pd.read_csv(CSV_FILE)
if "gamma" in df.columns and "gamma_final" not in df.columns:
    df = df.rename(columns={"gamma": "gamma_final"})
df["angle_deg"] = df["angle_deg"].round().astype(int)

angles = sorted(df["angle_deg"].unique())

for a in ALPHAS:
    d = df[df["alpha"] == a]
    if len(d) == 0:
        print(f"[WARN] alpha = {a:.0e} not in CSV")
        continue

    rows = []
    for ang in angles:
        dd = d[d.angle_deg == ang]
        N = len(dd)
        n_ej = int((dd.status == "Ejected").sum())
        n_tr = int((dd.status == "Trapped").sum())
        n_cr = int((dd.status == "Crashed").sum())
        rows.append({
            "chi": ang,
            "N": N,
            "Ej_n": n_ej, "Ej_pct": 100*n_ej/max(N,1),
            "Tr_n": n_tr, "Tr_pct": 100*n_tr/max(N,1),
            "Cr_n": n_cr, "Cr_pct": 100*n_cr/max(N,1),
        })
    tbl = pd.DataFrame(rows)
    print("=" * 90)
    print(f"FATE FRACTIONS — alpha = {a:.0e}")
    print("=" * 90)
    print(tbl.to_string(index=False, float_format="%.2f"))
    print()

FATE FRACTIONS — alpha = 1e+02
 chi    N  Ej_n  Ej_pct  Tr_n  Tr_pct  Cr_n  Cr_pct
   0 1000   903   90.30    87    8.70    10    1.00
  15 1000   912   91.20    81    8.10     7    0.70
  30 1000   937   93.70    53    5.30    10    1.00
  45 1000   960   96.00    34    3.40     6    0.60
  60 1000   970   97.00    20    2.00    10    1.00
  75 1000   946   94.60    38    3.80    16    1.60
  90 1000   895   89.50    78    7.80    27    2.70
 105 1000   786   78.60   144   14.40    70    7.00
 120 1000   678   67.80   220   22.00   102   10.20
 135 1000   545   54.50   322   32.20   133   13.30
 150 1000   383   38.30   447   44.70   170   17.00
 165 1000   224   22.40   559   55.90   217   21.70
 180 1000     0    0.00   845   84.50   155   15.50

FATE FRACTIONS — alpha = 1e+03
 chi    N  Ej_n  Ej_pct  Tr_n  Tr_pct  Cr_n  Cr_pct
   0 1000   708   70.80   282   28.20    10    1.00
  15 1000   736   73.60   257   25.70     7    0.70
  30 1000   799   79.90   191   19.10    10    1.00
 

In [5]:
# =========================================================
# SUMMARY METRICS — plateau, transition, minima, maxima
# =========================================================
import numpy as np
import pandas as pd

CSV_FILE = "particles_all_runs.csv"
ALPHAS   = [1e2, 1e3, 1e4,1e5]

df = pd.read_csv(CSV_FILE)
if "gamma" in df.columns and "gamma_final" not in df.columns:
    df = df.rename(columns={"gamma": "gamma_final"})
df["angle_deg"] = df["angle_deg"].round().astype(int)

print("=" * 90)
print("SUMMARY METRICS PER ALPHA")
print("=" * 90)

for a in ALPHAS:
    d = df[df["alpha"] == a]
    if len(d) == 0:
        continue

    piv = (d.groupby(["angle_deg", "status"])
             .size().unstack(fill_value=0))
    for col in ("Ejected", "Trapped", "Crashed"):
        if col not in piv.columns:
            piv[col] = 0
    piv["N"] = piv.sum(axis=1)
    piv["Ej_pct"] = 100 * piv["Ejected"] / piv["N"]
    piv["Tr_pct"] = 100 * piv["Trapped"] / piv["N"]
    piv["Cr_pct"] = 100 * piv["Crashed"] / piv["N"]

    # Plateau range (small tilt): first 5 angles 0 to 60
    small = piv.loc[piv.index <= 60]
    plateau_min = small["Ej_pct"].min()
    plateau_max = small["Ej_pct"].max()
    plateau_mean = small["Ej_pct"].mean()

    # Transition: find the angle with the largest drop in Ej_pct
    dp = piv["Ej_pct"].diff().abs()
    i_drop = dp.idxmax() if len(dp.dropna()) else None

    # Where does Ej_pct fall below 50%
    below50 = piv.index[piv["Ej_pct"] < 50]
    first_below50 = below50.min() if len(below50) else None

    # Where does Ej_pct go to zero
    zeros = piv.index[piv["Ej_pct"] < 1.0]
    first_zero = zeros.min() if len(zeros) else None

    # Peak of crashed fraction
    peak_crash_angle = piv["Cr_pct"].idxmax()
    peak_crash_value = piv["Cr_pct"].max()

    # Behavior at chi = 180
    chi180 = piv.loc[180] if 180 in piv.index else None

    print(f"\nalpha = {a:.0e}")
    print(f"  Plateau (chi <= 60):   Ej = {plateau_min:.1f}% to {plateau_max:.1f}%"
          f"  (mean {plateau_mean:.1f}%)")
    print(f"  Largest single drop:   at chi = {i_drop}°,"
          f"  drop of {dp.loc[i_drop]:.1f} pp")
    print(f"  First Ej < 50%:        chi = {first_below50}°")
    print(f"  First Ej < 1%:         chi = {first_zero}°")
    print(f"  Crash peak:            chi = {peak_crash_angle}°,"
          f"  {peak_crash_value:.1f}%")
    if chi180 is not None:
        print(f"  chi = 180°:            "
              f"Ej {chi180['Ej_pct']:.1f}%  "
              f"Tr {chi180['Tr_pct']:.1f}%  "
              f"Cr {chi180['Cr_pct']:.1f}%")

SUMMARY METRICS PER ALPHA

alpha = 1e+02
  Plateau (chi <= 60):   Ej = 90.3% to 97.0%  (mean 93.6%)
  Largest single drop:   at chi = 180°,  drop of 22.4 pp
  First Ej < 50%:        chi = 150°
  First Ej < 1%:         chi = 180°
  Crash peak:            chi = 165°,  21.7%
  chi = 180°:            Ej 0.0%  Tr 84.5%  Cr 15.5%

alpha = 1e+03
  Plateau (chi <= 60):   Ej = 70.8% to 88.0%  (mean 79.5%)
  Largest single drop:   at chi = 105°,  drop of 17.9 pp
  First Ej < 50%:        chi = 120°
  First Ej < 1%:         chi = 180°
  Crash peak:            chi = 165°,  57.0%
  chi = 180°:            Ej 0.0%  Tr 43.4%  Cr 56.6%

alpha = 1e+04
  Plateau (chi <= 60):   Ej = 68.8% to 84.3%  (mean 75.6%)
  Largest single drop:   at chi = 105°,  drop of 19.5 pp
  First Ej < 50%:        chi = 120°
  First Ej < 1%:         chi = 180°
  Crash peak:            chi = 165°,  58.7%
  chi = 180°:            Ej 0.0%  Tr 41.9%  Cr 58.1%

alpha = 1e+05
  Plateau (chi <= 60):   Ej = 68.3% to 83.4%  (mean 75.0%)


In [6]:
# =========================================================
# MAX LORENTZ FACTOR per angle, per alpha (ejected only)
# =========================================================
import numpy as np
import pandas as pd

CSV_FILE = "particles_all_runs.csv"
ALPHAS   = [1e2, 1e3, 1e4,1e5]

df = pd.read_csv(CSV_FILE)
if "gamma" in df.columns and "gamma_final" not in df.columns:
    df = df.rename(columns={"gamma": "gamma_final"})
df["angle_deg"] = df["angle_deg"].round().astype(int)
ej = df[df.status == "Ejected"]

angles = sorted(df["angle_deg"].unique())

rows = []
for a in ALPHAS:
    d = ej[ej["alpha"] == a]
    row = {"alpha": a}
    for ang in angles:
        dd = d[d.angle_deg == ang]
        row[f"gmax_{ang}"] = dd.gamma_final.max() if len(dd) else np.nan
    rows.append(row)

tbl = pd.DataFrame(rows)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.1f}'.format)
print("=" * 120)
print("MAX GAMMA per angle, per alpha (ejected particles only)")
print("=" * 120)
print(tbl.to_string(index=False))

# Ratio check for scaling
print("\n" + "=" * 90)
print("SCALING CHECK — ratio of median gamma between successive alphas")
print("=" * 90)
for a_lo, a_hi in [(1e2, 1e3), (1e3, 1e4)]:
    d_lo = ej[ej["alpha"] == a_lo]
    d_hi = ej[ej["alpha"] == a_hi]
    if len(d_lo) == 0 or len(d_hi) == 0:
        continue
    print(f"\nalpha_hi / alpha_lo = {a_hi/a_lo:.1f}")
    print(f"{'chi':>5}  {'med_lo':>10}  {'med_hi':>10}  {'ratio':>8}")
    for ang in angles:
        g_lo = d_lo[d_lo.angle_deg == ang].gamma_final
        g_hi = d_hi[d_hi.angle_deg == ang].gamma_final
        if len(g_lo) < 5 or len(g_hi) < 5:
            continue
        m_lo, m_hi = np.median(g_lo), np.median(g_hi)
        print(f"{ang:>5}  {m_lo:>10.2f}  {m_hi:>10.2f}  {m_hi/m_lo:>8.3f}")

MAX GAMMA per angle, per alpha (ejected particles only)
   alpha  gmax_0  gmax_15  gmax_30  gmax_45  gmax_60  gmax_75  gmax_90  gmax_105  gmax_120  gmax_135  gmax_150  gmax_165  gmax_180
   100.0    10.0     11.6     12.3     13.5     12.6     13.0     13.4      12.2      10.3       8.2       6.0       3.8       NaN
  1000.0    91.2    106.1    108.9    112.4    113.9    117.6    111.9     112.1      88.9      71.9      47.8      31.5       NaN
 10000.0   903.5   1048.4   1063.1   1083.2   1126.6   1173.7   1108.3    1106.9     881.9     678.3     466.1     313.5       NaN
100000.0  9025.4  10470.2  10601.7  10818.9  11263.1  11735.5  11069.7   11051.2    8806.4    6706.7    4628.1    3051.1       NaN

SCALING CHECK — ratio of median gamma between successive alphas

alpha_hi / alpha_lo = 10.0
  chi      med_lo      med_hi     ratio
    0        3.99       30.24     7.586
   15        3.99       30.04     7.534
   30        3.96       28.46     7.190
   45        3.68       23.64     6.